# Semantic Search with Vector Engine\nRAG-style local retrieval example.

In [ ]:
import numpy as np\nfrom sentence_transformers import SentenceTransformer\nfrom vector_engine import VectorArray, VectorIndex\n\nmodel = SentenceTransformer(\"all-MiniLM-L6-v2\")\n\ndocs = [\n    \"Vector search can accelerate semantic retrieval.\",\n    \"KNN is a strong non-parametric baseline.\",\n    \"Faiss supports fast similarity search at scale.\",\n    \"Cosine similarity is common for normalized embeddings.\",\n    \"Hard negatives improve contrastive training signals.\",\n]\n\ndoc_emb = model.encode(docs, normalize_embeddings=True).astype(\"float32\")\nmetadata = [{\"text\": t} for t in docs]\nxb = VectorArray.from_numpy(doc_emb, ids=[f\"doc-{i}\" for i in range(len(docs))], metadata=metadata)\n\nindex = VectorIndex.create(\n    xb,\n    metric=\"cosine\",\n    backend=\"faiss\",\n    backend_config={\"index_factory\": \"Flat\"},\n)\n\nquery = \"How do I retrieve similar text quickly?\"\nq_emb = model.encode([query], normalize_embeddings=True).astype(\"float32\")\nxq = VectorArray.from_numpy(q_emb)\nres = index.search(xq, k=3, return_metadata=True)\n\nfor rank, (doc_id, score, md) in enumerate(zip(res.ids[0], res.scores[0], res.metadata[0]), 1):\n    print(f\"{rank}. {doc_id} score={score:.4f} text={md['text']}\")\n